# 14 — Data augmentation a codec e fine-tuning LP-FT (P, Wf, E, PWfE)

**Domanda.** L'augmentation con codec multimediali, la tecnica di preprocessamento audio del vincitore
del DF (T23: «Media Codec» in Tab. V e Fig. 11 di Liu et al. 2023, arXiv:2210.02437), migliora i quattro
modelli su ASVspoof 2021 DF `eval`?

**Disegno (scelte registrate prima dell'esecuzione)**

| | configurazione | dati di addestramento | semi |
|---|---|---|---|
| 0 | checkpoint esistenti `model_{D,Wf,E,PWfE}_seed0.pt` | — | 1 (riferimento storico) |
| 1 | LP-FT di **controllo** dal checkpoint | 25.380 originali | 3 |
| 2 | LP-FT con **augmentation** dal checkpoint | originali + 1 copia per codec (mp3, m4a, ogg) = 101.520 | 3 |

**L'effetto dell'augmentation è il confronto 2 contro 1**, non 2 contro 0: anche il solo addestramento
aggiuntivo dal checkpoint può cambiare l'EER.

**LP-FT** (Kumar et al., *Fine-Tuning can Distort Pretrained Features and Underperform
Out-of-Distribution*, ICLR 2022, arXiv:2202.10054):
- **fase LP**: solo la testa (`head`; per PWfE anche `bn_fuse`), rami congelati **e in `eval()`**, così
  che nemmeno le statistiche delle loro BatchNorm cambino; lr 1e-3, come l'addestramento originale;
- **fase FT**: tutti i parametri, a partire dal risultato della fase LP; lr 1e-4 (fattore 10: convenzione
  fissata a priori, non ottimizzata sul dev).

In entrambe le fasi: Adam, batch 32, BCE pesata (`pos_weight` = 8,8372), `ReduceLROnPlateau` (0,5, pazienza 3)
ed early stopping (pazienza 10, massimo 40 epoche) sull'**EER del dev** 2019 pulito, come nel notebook 10.
Si applica sempre almeno un'epoca: il checkpoint di partenza non è un candidato alla selezione.

**Augmentation.** Il paper non dice quali codec né quali bitrate abbia usato T23. Le scelte sono nostre,
fatte sulla **Tab. IV** (condizioni dichiarate del DF), **non** sugli EER per codec ottenuti su `eval`:
mp3 (libmp3lame), m4a (AAC nativo di ffmpeg), ogg (libvorbis); per ogni file e codec un livello `low`/`high`
e un bitrate estratti a caso, con un generatore derivato da `utt_id` e codec. Il «chunking» di T23 non si usa:
l'architettura gestisce lunghezze variabili.

**Fasi** (`FASE`, una o più separate da `+`; ogni fase riprende da dove si era fermata):
1. `verifica` — encoder disponibili, effetto misurato dei codec su file reali, **test di equivalenza**:
   le caratteristiche ricalcolate con il nuovo percorso sugli originali devono riprodurre quelle salvate;
2. `estrazione` — copie aumentate del train: Praat, Whisper, ECAPA (circa 4 ore);
3. `addestramento` — controllo di riproduzione dei checkpoint sul dev, poi LP-FT;
4. `valutazione` — controllo di riproduzione su `eval`, punteggi, confronti appaiati.

**Ipotesi registrate prima dell'esecuzione** (*ragionamento sui dati della tesi, non risultati*):
- l'effetto su `eval` sarà piccolo: fra le 9 condizioni di codec l'escursione di PWfE è di 1,03 punti,
  contro 18,04 fra le 3 sorgenti;
- se l'augmentation agisce, l'escursione fra le 9 condizioni di codec dovrebbe ridursi.

**Regola di decisione.** Per ogni modello e fase: IC 95% del bootstrap appaiato stratificato di
δ = EER(aug) − EER(controllo), calcolato **per ciascun seme**. Si conclude «augmentation migliore» solo se
l'IC è interamente sotto lo zero **in tutti e tre i semi**; «peggiore» se interamente sopra in tutti e tre;
altrimenti «non conclusivo».

**Metriche.** EER con `eval_metrics.py` ufficiale ASVspoof 2021 (SHA256 `70c5d2fe8fab6654…`).

In [1]:
import os, sys, re, json, math, time, copy, hashlib, importlib.util, urllib.request
import subprocess, tempfile, shutil
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import pandas as pd
import soundfile as sf
from scipy import signal as sps
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, PackedSequence

# ================================================================================
FASE = os.environ.get("FASE", "addestramento+valutazione")
# Una o piu' fasi separate da '+', eseguite in quest'ordine: verifica | estrazione | addestramento | valutazione
# Consigliato: "verifica+estrazione" in una sessione (se la verifica fallisce l'esecuzione si ferma prima
# dell'estrazione), poi "addestramento+valutazione" in un'altra.
# ================================================================================
ATTIVE = set(FASE.split("+"))
assert ATTIVE <= {"verifica", "estrazione", "addestramento", "valutazione"}, FASE
# --- PERCORSI (slug verificati nei notebook 08, 10 e 11) ---------------------
BASE_A = Path(os.environ.get("BASE_A", "/kaggle/input/datasets/angelopaldino"))
BASE_B = Path("/kaggle/input/datasets/angelo01paldino")
LA_ROOT       = BASE_A / "asvspoof2019-train-dev/LA"                    # audio 2019 (notebook 08)
PROSODY_DIR   = BASE_A / "prosody-features/prosody_features"            # notebook 04
PTM_TRAIN_DIR = BASE_A / "outputtrain-feature-noft/ptm_features_train"  # notebook 08, train
PTM_DEV_DIR   = BASE_A / "devroot/root"                                 # notebook 08, dev (come nel 10)
MODELLI_DIR   = BASE_A / "models/Modelli"
DF_KEYS       = BASE_A / "df-keys/CM/trial_metadata.txt"
DF_PTM_DIR    = BASE_A / "features/resultsPTM/df2021/ptm"               # eval, 533.928 (notebook 11)
DF_PROS_DIR   = BASE_A / "features/results_Prosodia/df2021/prosodia"    # eval, 533.928 (notebook 11)
PREPROC_PROSODIA = MODELLI_DIR / "preproc_prosody.json"
STANDARDIZER     = MODELLI_DIR / "standardizer.json"
CKPT = {"P": MODELLI_DIR / "model_D_seed0.pt", "Wf": MODELLI_DIR / "model_Wf_seed0.pt",
        "E": MODELLI_DIR / "model_E_seed0.pt", "PWfE": MODELLI_DIR / "model_PWfE_seed0.pt"}

OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working/aug_lpft"))     # scrittura
# Ripresa: percorso, sotto /kaggle/input, della cartella 'aug_lpft' salvata da una sessione precedente.
# Deve contenere direttamente le sottocartelle 'caratteristiche_aug', 'modelli', ... Vuoto = nessuna ripresa.
RIPRESA = os.environ.get("RIPRESA", "/kaggle/input/datasets/angelo01paldino/dataft1/aug_lpft")

AUG_DIR = OUT_DIR / "caratteristiche_aug"
MOD_DIR = OUT_DIR / "modelli"
SCO_DIR = OUT_DIR / "punteggi_eval"
TMP_DIR = OUT_DIR / "tmp"

# --- audio ed estrattori, identici al notebook 08 -----------------------------
SR, HOP_SAMPLES, MAX_FRAMES = 16000, 320, 1500
WHISPER_MODEL = "openai/whisper-base"
ECAPA_MODEL   = "speechbrain/spkrec-ecapa-voxceleb"
WHISPER_BATCH, ECAPA_BATCH = 16, 16
BLOCCO = 2000              # = SHARD_FILES del notebook 08: stessi file nello stesso batch
N_WORKERS = os.cpu_count() or 1

# --- Praat, identici al notebook 04 (configurazione 'standard') -------------
PITCH_FLOOR, PITCH_CEILING = 75.0, 500.0
MAX_CANDIDATES, VERY_ACCURATE, VOICING_THRESHOLD, TIME_STEP = 15, "no", 0.45, 0.0
PERIOD_FLOOR, PERIOD_CEILING = 0.8 / PITCH_CEILING, 1.25 / PITCH_FLOOR
MAX_PERIOD_FACTOR, MAX_AMPLITUDE_FACTOR = 1.3, 1.6
HNR_TIME_STEP, HNR_SILENCE, HNR_PERIODS = 0.01, 0.1, 1.0
WINDOW_S = 0.2
CFG_STANDARD = {"silence_threshold": 0.03, "octave_cost": 0.01,
                "octave_jump_cost": 0.35, "voiced_unvoiced_cost": 0.14}
FEATURES = ["f0_mean", "f0_std", "jitter_local", "shimmer_local",
            "hnr_mean", "hnr_std", "intensity_mean", "intensity_std"]

# --- augmentation a codec ----------------------------------------------------
SEED = 0
CODECS = ["mp3", "m4a", "ogg"]
# Frequenza di codifica. A 16 kHz mono gli intervalli della Tab. IV non sono raggiungibili (misurato prima
# di scrivere il notebook, su un segnale di prova: mp3 si ferma a ~160 kbps, il limite di MPEG-2; AAC a ~78;
# Vorbis a ~80 con -q:a 10). Si codifica quindi a 44,1 kHz e si decodifica di nuovo a 16 kHz.
ENC_SR = 44100
# livello: (modo, minimo, massimo). modo 'b' = bitrate richiesto in kbps; 'q' = qualita' VBR.
# ogg 'high' (256-320 kbps in Tab. IV) non e' ottenibile in mono con libvorbis (encoder setup failed a 256k):
# si usa la qualita' massima -q:a 10. Il bitrate EFFETTIVO di ogni copia viene misurato e salvato.
CODEC_CFG = {
    "mp3": {"enc": "libmp3lame", "ext": "mp3", "livelli": {"low": ("b", 80, 120), "high": ("b", 220, 260)}},
    "m4a": {"enc": "aac",        "ext": "m4a", "livelli": {"low": ("b", 20, 32),  "high": ("b", 96, 112)}},
    "ogg": {"enc": "libvorbis",  "ext": "ogg", "livelli": {"low": ("b", 80, 96),  "high": ("q", 10, 10)}},
}
N_PROVE_CODEC = 30         # file reali per la misura dell'effetto dei codec (fase 'verifica')

# --- modelli e addestramento -------------------------------------------------
D_WHISPER, D_ECAPA, EMB_DIM, DROPOUT = 512, 192, 50, 0.2
N_CH = 2 * len(FEATURES)                     # variante D: 8 caratteristiche + delta
NOMI = ["P", "Wf", "E", "PWfE"]
CONDIZIONI_FT = ["controllo", "aug"]
SEEDS = [0, 1, 2]
FASI_LPFT = ["LP", "FT"]
LR = {"LP": 1e-3, "FT": 1e-4}
BATCH_SIZE, MAX_EPOCHS, PATIENCE = 32, 40, 10
LR_FACTOR, LR_PATIENCE = 0.5, 3
EVAL_BATCH = 256
N_BOOT = int(os.environ.get("N_BOOT", 1000))
MIN_PER_CLASSE = 10
# parametri addestrabili attesi: servono a verificare che il congelamento sia quello dichiarato
N_PARAM_LP = {"P": 2601, "Wf": 2601, "E": 2601, "PWfE": 7901}
N_PARAM_FT = {"P": 239645, "Wf": 74971, "E": 34011, "PWfE": 348725}

# --- riferimenti per i controlli di riproduzione -----------------------------
# EER del dev 2019 stampati dai notebook di addestramento (06.1 variante D seme 0, 09 E, 10 PWfE).
# Per Wf il notebook di addestramento non e' disponibile: il valore viene solo stampato.
EER_DEV_RIF  = {"P": 11.77, "E": 9.06, "PWfE": 1.29}
# EER su ASVspoof 2021 DF eval (533.928) stampati dal notebook 11 con gli stessi checkpoint e caratteristiche
EER_EVAL_RIF = {"P": 26.83, "Wf": 12.35, "E": 22.09, "PWfE": 16.09}
TOLL_RIF = 0.05            # punti di EER: i valori di riferimento sono arrotondati a 2 decimali
# Test di equivalenza. Il criterio sui soli punteggi NON basta: nel collaudo uno spostamento sistematico del 5%
# dell'intensita' media su tutti i file lo superava. Servono anche criteri sulle caratteristiche.
TOLL_MEDIANA = 1e-2          # scarto mediano dei punteggi (Sez. 6.7)
QUOTA_PRAAT_IDENTICI = 0.99  # Praat e' deterministico: stesso segnale e stessa versione -> stessi valori
TOLL_REL_PTM = 1e-3          # norma relativa mediana di whisper_mean ed ecapa (ammette solo rumore numerico)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

for d in (OUT_DIR, AUG_DIR, MOD_DIR, SCO_DIR, TMP_DIR):
    d.mkdir(parents=True, exist_ok=True)
_prova = OUT_DIR / ".prova_scrittura"          # fallire subito, non dopo ore di calcolo
_prova.write_text("ok"); _prova.unlink()

# --- ripresa -----------------------------------------------------------------
if RIPRESA:
    src = Path(RIPRESA)
    assert src.exists(), f"RIPRESA non esiste: {src}. Verifica con  !find /kaggle/input -maxdepth 6 -name aug_lpft"
    attese = [p for p in ("caratteristiche_aug", "modelli", "punteggi_eval") if (src / p).is_dir()]
    assert attese, (f"{src} non contiene 'caratteristiche_aug', 'modelli' o 'punteggi_eval': "
                    f"RIPRESA deve puntare alla cartella 'aug_lpft', non a una sua sottocartella")
    shutil.copytree(src, OUT_DIR, dirs_exist_ok=True)
    for p in attese:
        print(f"ripristinati da {src / p}: {len(list((OUT_DIR / p).iterdir()))} file")
else:
    print("nessuna ripresa (RIPRESA vuoto)")

print(f"FASE = {FASE} | device {DEVICE} | core {N_WORKERS} | torch {torch.__version__} | output {OUT_DIR}")

ripristinati da /kaggle/input/datasets/angelo01paldino/dataft1/aug_lpft/caratteristiche_aug: 39 file
FASE = addestramento+valutazione | device cuda | core 4 | torch 2.10.0+cu128 | output /kaggle/working/aug_lpft


In [2]:
# Metrica ufficiale ASVspoof 2021, identica ai notebook precedenti.
EVAL_METRICS_URL = "https://raw.githubusercontent.com/asvspoof-challenge/2021/main/eval-package/eval_metrics.py"
EVAL_METRICS_PATH = OUT_DIR / "eval_metrics.py"
if not EVAL_METRICS_PATH.exists():
    urllib.request.urlretrieve(EVAL_METRICS_URL, EVAL_METRICS_PATH)
spec = importlib.util.spec_from_file_location("asvspoof_eval_metrics", EVAL_METRICS_PATH)
asv_metrics = importlib.util.module_from_spec(spec); spec.loader.exec_module(asv_metrics)
EVAL_METRICS_SHA256 = hashlib.sha256(EVAL_METRICS_PATH.read_bytes()).hexdigest()
assert EVAL_METRICS_SHA256.startswith("70c5d2fe8fab6654"), f"eval_metrics.py inatteso: {EVAL_METRICS_SHA256[:16]}"
print("eval_metrics.py | sha256", EVAL_METRICS_SHA256[:16], "...")


def compute_eer(y, s):
    # y = 1 bona fide, 0 spoof; s alto = bona fide
    y, s = np.asarray(y), np.asarray(s, dtype=np.float64)
    return float(asv_metrics.compute_eer(s[y == 1], s[y == 0])[0])


def bootstrap_appaiato(y, s_rif, s_alt, n_boot=None, seed=SEED):
    """IC 95% di EER(s_alt) - EER(s_rif) in punti, come nel notebook 13.

    Ricampiona separatamente bona fide e spoof; i due sistemi valutano gli STESSI file in ogni replica.
    """
    n_boot = n_boot or N_BOOT
    g = np.random.default_rng(seed)
    i_bf, i_sp = np.where(y == 1)[0], np.where(y == 0)[0]
    d = np.empty(n_boot)
    for k in range(n_boot):
        idx = np.concatenate([g.choice(i_bf, len(i_bf)), g.choice(i_sp, len(i_sp))])
        yy = y[idx]
        d[k] = compute_eer(yy, s_alt[idx]) - compute_eer(yy, s_rif[idx])
    d *= 100
    return float(d.mean()), tuple(np.percentile(d, [2.5, 97.5]))


assert compute_eer([1, 1, 0, 0], [2.0, 3.0, 0.0, 1.0]) == 0.0
assert compute_eer([1, 1, 0, 0], [0.0, 1.0, 2.0, 3.0]) == 1.0

eval_metrics.py | sha256 70c5d2fe8fab6654 ...


In [3]:
# Protocolli ASVspoof 2019 LA, letti come nel notebook 08: l'ordine dei file e' quello del protocollo.
PATTERNS = {"train": r"cm[._]train[._]trn", "dev": r"cm[._]dev[._]trl"}
EXPECTED_ROWS = {"train": 25380, "dev": 24844}


def leggi_protocollo(split):
    f = [p for p in (LA_ROOT / "ASVspoof2019_LA_cm_protocols").iterdir() if re.search(PATTERNS[split], p.name)]
    assert len(f) == 1, f
    df = pd.read_csv(f[0], sep=r"\s+", header=None, dtype=str, keep_default_na=False, engine="python",
                     names=["speaker", "utt_id", "unused", "attack", "label"]).drop(columns="unused")
    df["attack"] = df["attack"].replace("-", "bonafide")
    df["y_bonafide"] = (df["label"] == "bonafide").astype(np.int8)
    df["path"] = [str(LA_ROOT / f"ASVspoof2019_LA_{split}" / "flac" / f"{u}.flac") for u in df["utt_id"]]
    assert len(df) == EXPECTED_ROWS[split], (split, len(df))
    return df


assert (LA_ROOT / "ASVspoof2019_LA_cm_protocols").exists(), f"LA_ROOT errato: {LA_ROOT}"
FILES = {s: leggi_protocollo(s) for s in ["train", "dev"]}
for s, df in FILES.items():
    print(f"{s}: {len(df)} file | bona fide {int(df['y_bonafide'].sum())}")
if ATTIVE & {"verifica", "estrazione"}:
    manca = [p for p in FILES["train"]["path"] if not os.path.exists(p)]
    assert not manca, manca[:5]


def carica_prosodia(split):
    """Sequenze Praat salvate dal notebook 04 (configurazione 'standard'), nell'ordine del protocollo."""
    z = np.load(PROSODY_DIR / f"standard_{split}.npz", allow_pickle=True)
    X, off, utt = z["X"].astype(np.float32), z["offsets"].astype(np.int64), z["utt_id"]
    assert (utt == FILES[split]["utt_id"].to_numpy()).all(), f"prosodia {split}: ordine diverso dal protocollo"
    assert int(off[-1]) == len(X) and np.isfinite(X).all()
    return [X[off[i]:off[i + 1]] for i in range(len(utt))]


def carica_ptm(split):
    """whisper_mean ed ecapa salvati dal notebook 08, nell'ordine del protocollo."""
    d = PTM_TRAIN_DIR if split == "train" else PTM_DEV_DIR
    zw = np.load(d / f"whisper_mean_{split}.npz", allow_pickle=True)
    ze = np.load(d / f"ecapa_{split}.npz", allow_pickle=True)
    ref = FILES[split]["utt_id"].to_numpy()
    assert (zw["utt_id"] == ref).all() and (ze["utt_id"] == ref).all(), f"PTM {split}: ordine diverso dal protocollo"
    wm, ec = zw["whisper_mean"].astype(np.float32), ze["ecapa"].astype(np.float32)
    assert wm.shape == (len(ref), D_WHISPER) and ec.shape == (len(ref), D_ECAPA)
    assert np.isfinite(wm).all() and np.isfinite(ec).all()
    return wm, ec

train: 25380 file | bona fide 2580
dev: 24844 file | bona fide 2548


In [4]:
# --- Praat: funzioni identiche al notebook 04, applicate al segnale in memoria --------------------------
# Il notebook 04 apriva il .flac con parselmouth.Sound(path). Qui il segnale e' letto con soundfile, come nel
# notebook 08, e passato a Praat come array: il test di equivalenza della fase 'verifica' controlla che il
# risultato coincida con le caratteristiche salvate.
try:
    import parselmouth
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "praat-parselmouth==0.4.7"], check=True)
    import parselmouth
from parselmouth.praat import call
# Le caratteristiche salvate (notebook 04) sono state estratte con parselmouth 0.4.7 / Praat 6.1.38: una
# versione diversa puo' cambiarle. Se questo assert fallisce, aggiungi in cima al notebook
# !pip install -q praat-parselmouth==0.4.7  e riavvia la sessione.
assert (parselmouth.__version__, parselmouth.PRAAT_VERSION) == ("0.4.7", "6.1.38"), \
    f"parselmouth {parselmouth.__version__} / Praat {parselmouth.PRAAT_VERSION}: attesi 0.4.7 / 6.1.38"


def make_windows(duration, w):
    starts = np.arange(0.0, duration, w)
    ends = np.minimum(starts + w, duration)
    keep = (ends - starts) >= w / 2
    return starts[keep], ends[keep]


def window_features(snd, pp, ht, hv, it, iv, a, b, pt, f0):
    f0w = f0[(pt >= a) & (pt < b)]
    f0w = f0w[f0w > 0]
    if len(f0w) == 0:
        return np.zeros(len(FEATURES))
    out = np.full(len(FEATURES), np.nan)
    out[0] = f0w.mean()
    out[1] = f0w.std() if len(f0w) >= 2 else np.nan
    try:
        out[2] = call(pp, "Get jitter (local)", a, b, PERIOD_FLOOR, PERIOD_CEILING, MAX_PERIOD_FACTOR)
    except Exception:
        pass
    try:
        out[3] = call([snd, pp], "Get shimmer (local)", a, b, PERIOD_FLOOR, PERIOD_CEILING,
                      MAX_PERIOD_FACTOR, MAX_AMPLITUDE_FACTOR)
    except Exception:
        pass
    hw = hv[(ht >= a) & (ht < b)]; hw = hw[hw > -100]
    if len(hw):
        out[4] = hw.mean(); out[5] = hw.std() if len(hw) >= 2 else np.nan
    iw = iv[(it >= a) & (it < b)]; iw = iw[np.isfinite(iw)]
    if len(iw):
        out[6] = iw.mean(); out[7] = iw.std() if len(iw) >= 2 else np.nan
    return np.nan_to_num(out, nan=0.0)


def praat_da_segnale(x):
    snd = parselmouth.Sound(np.asarray(x, dtype=np.float64), sampling_frequency=SR)
    harm = call(snd, "To Harmonicity (cc)", HNR_TIME_STEP, PITCH_FLOOR, HNR_SILENCE, HNR_PERIODS)
    inten = snd.to_intensity(minimum_pitch=PITCH_FLOOR)
    ht, hv = np.asarray(harm.xs()), np.asarray(harm.values[0])
    it, iv = np.asarray(inten.xs()), np.asarray(inten.values[0])
    starts, ends = make_windows(snd.duration, WINDOW_S)
    pitch = call(snd, "To Pitch (cc)", TIME_STEP, PITCH_FLOOR, MAX_CANDIDATES, VERY_ACCURATE,
                 CFG_STANDARD["silence_threshold"], VOICING_THRESHOLD, CFG_STANDARD["octave_cost"],
                 CFG_STANDARD["octave_jump_cost"], CFG_STANDARD["voiced_unvoiced_cost"], PITCH_CEILING)
    pp = call([snd, pitch], "To PointProcess (cc)")
    pt, f0 = np.asarray(pitch.xs()), np.asarray(pitch.selected_array["frequency"])
    X = np.zeros((max(len(starts), 1), len(FEATURES)), dtype=np.float32)
    for i, (a, b) in enumerate(zip(starts, ends)):
        X[i] = window_features(snd, pp, ht, hv, it, iv, a, b, pt, f0)
    return X


# --- codec ---------------------------------------------------------------------------------------------
def leggi_audio(path):
    # identico al notebook 08
    x, sr = sf.read(path, dtype="float32", always_2d=False)
    assert sr == SR and x.ndim == 1, (path, sr, x.shape)
    return x


def _allinea(y, n):
    if len(y) >= n:
        return np.ascontiguousarray(y[:n], dtype=np.float32)
    return np.pad(y, (0, n - len(y))).astype(np.float32)


def _togli_ritardo(y, x, max_lag=4096, n_corr=SR * 2):
    """Come nel notebook 13: stima per correlazione incrociata il ritardo di y su x e lo rimuove."""
    a = x[:n_corr].astype(np.float64); b = y[:n_corr + max_lag].astype(np.float64)
    if len(b) <= len(a) or np.allclose(a, 0) or np.allclose(b, 0):
        return y, 0
    lag = int(np.argmax(sps.correlate(b, a, mode="valid")))
    return (y[lag:], lag) if lag > 0 else (y, lag)


def _ffmpeg(args):
    r = subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", *args], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"ffmpeg: {r.stderr[:300]}")


def codifica(x, codec, utt):
    """Codifica e decodifica x con il codec dato. Livello e bitrate dipendono solo da (SEED, utt_id, codec):
    la stessa copia si ottiene in qualunque ordine e dopo qualunque ripresa (Sez. 6.9)."""
    cfg = CODEC_CFG[codec]
    g = np.random.default_rng([SEED, int(hashlib.sha256(f"{utt}|{codec}".encode()).hexdigest()[:8], 16)])
    livello = ["low", "high"][int(g.integers(2))]
    modo, lo, hi = cfg["livelli"][livello]
    val = int(g.integers(lo, hi + 1))
    opt = ["-b:a", f"{val}k"] if modo == "b" else ["-q:a", str(val)]
    with tempfile.TemporaryDirectory(dir=TMP_DIR) as d:
        pin, penc, pout = Path(d) / "i.wav", Path(d) / f"e.{cfg['ext']}", Path(d) / "o.wav"
        sf.write(pin, x, SR, subtype="PCM_16")                 # l'originale e' FLAC a 16 bit: senza perdita
        _ffmpeg(["-i", str(pin), "-ar", str(ENC_SR), "-ac", "1", "-c:a", cfg["enc"], *opt, str(penc)])
        kbps = penc.stat().st_size * 8 / (len(x) / SR) / 1000     # bitrate effettivo, contenitore incluso
        _ffmpeg(["-i", str(penc), "-ar", str(SR), "-ac", "1", "-c:a", "pcm_s16le", str(pout)])
        y, sr = sf.read(pout, dtype="float32")
        assert sr == SR and y.ndim == 1
    y, lag = _togli_ritardo(y, x)
    y = _allinea(y, len(x))
    err = float(np.mean((y.astype(np.float64) - x) ** 2))
    snr = np.inf if err <= 0 else 10 * np.log10(float(np.mean(x.astype(np.float64) ** 2)) / err)
    return y, {"livello": livello, "param": " ".join(opt), "kbps": kbps, "lag": lag, "snr": snr}


def elabora_file(arg):
    """Lavoro di un processo: lettura, eventuale codec, Praat. codec None = identita' (test di equivalenza)."""
    path, utt, codec = arg
    try:
        x = leggi_audio(path)
        if codec is None:
            y, meta = x, {"livello": "-", "param": "-", "kbps": np.nan, "lag": 0, "snr": np.inf}
        else:
            y, meta = codifica(x, codec, utt)
        assert np.isfinite(y).all(), "campioni non finiti"
        return praat_da_segnale(y), y, meta, ""
    except Exception as e:
        return None, None, None, f"{utt}: {e!r}"


def elabora_blocco(paths, utts, codec):
    with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
        res = list(ex.map(elabora_file, [(p, u, codec) for p, u in zip(paths, utts)], chunksize=8))
    errori = [r[3] for r in res if r[3]]
    assert not errori, f"{len(errori)} file non elaborati, primi: {errori[:3]}"
    return [r[0] for r in res], [r[1] for r in res], [r[2] for r in res]

In [5]:
# --- estrattori, identici al notebook 08 (caricati solo nelle fasi che li usano) ------------------------
def carica_estrattori():
    # come nel notebook 08: installazione solo se mancano
    try:
        from transformers import WhisperModel, WhisperFeatureExtractor
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers"], check=True)
        from transformers import WhisperModel, WhisperFeatureExtractor
    try:
        from speechbrain.inference.speaker import EncoderClassifier
    except ImportError:
        try:
            from speechbrain.pretrained import EncoderClassifier
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "speechbrain"], check=True)
            from speechbrain.inference.speaker import EncoderClassifier
    import transformers, speechbrain
    print(f"transformers {transformers.__version__} | speechbrain {speechbrain.__version__}")
    feat_extractor = WhisperFeatureExtractor.from_pretrained(WHISPER_MODEL)
    whisper = WhisperModel.from_pretrained(WHISPER_MODEL).encoder.to(DEVICE).eval()
    for p in whisper.parameters():
        p.requires_grad_(False)
    ecapa = EncoderClassifier.from_hparams(source=ECAPA_MODEL, savedir=str(OUT_DIR / "_ecapa"),
                                           run_opts={"device": DEVICE})
    ecapa.eval()
    for p in ecapa.mods.parameters():
        p.requires_grad_(False)

    def n_frames_of(n_samples):
        return int(min(math.ceil(n_samples / HOP_SAMPLES), MAX_FRAMES))

    @torch.no_grad()
    def whisper_mean_batch(waves):
        feats = feat_extractor([w for w in waves], sampling_rate=SR, return_tensors="pt")
        out = whisper(feats.input_features.to(DEVICE)).last_hidden_state.float().cpu().numpy()
        return np.stack([out[i, :n_frames_of(len(w))].mean(0) for i, w in enumerate(waves)])

    @torch.no_grad()
    def ecapa_batch(waves):
        lens = torch.tensor([len(w) for w in waves], dtype=torch.float32)
        maxlen = int(lens.max())
        batch = torch.zeros(len(waves), maxlen)
        for i, w in enumerate(waves):
            batch[i, :len(w)] = torch.from_numpy(w)
        emb = ecapa.encode_batch(batch.to(DEVICE), (lens / maxlen).to(DEVICE))
        return emb.squeeze(1).float().cpu().numpy()

    return whisper_mean_batch, ecapa_batch


def estrai_ptm(waves, whisper_mean_batch, ecapa_batch):
    """Stessa suddivisione in batch del notebook 08: 16 file per Whisper, e al suo interno 16 per ECAPA."""
    wm, ec = [], []
    for i in range(0, len(waves), WHISPER_BATCH):
        w = waves[i:i + WHISPER_BATCH]
        wm.append(whisper_mean_batch(w))
        for j in range(0, len(w), ECAPA_BATCH):
            ec.append(ecapa_batch(w[j:j + ECAPA_BATCH]))
    return np.concatenate(wm).astype(np.float32), np.concatenate(ec).astype(np.float32)

In [6]:
# --- modelli: classi identiche ai notebook 11 e 13 (i nomi dei parametri corrispondono ai checkpoint) ---
def masked_bn(x, mask, bn):
    y = torch.zeros_like(x); y[mask] = bn(x[mask]); return y


def map_packed(p, fn):
    return PackedSequence(fn(p.data), p.batch_sizes, p.sorted_indices, p.unsorted_indices)


class SeqBranch(nn.Module):
    def __init__(self, n_in, p=DROPOUT):
        super().__init__()
        self.conv1, self.bn_c1 = nn.Conv1d(n_in, 256, kernel_size=3, padding=1), nn.BatchNorm1d(256)
        self.conv2, self.bn_c2 = nn.Conv1d(256, 128, kernel_size=3, padding=1), nn.BatchNorm1d(128)
        self.drop = nn.Dropout(p)
        self.lstm1, self.bn_l1 = nn.LSTM(128, 100, batch_first=True), nn.BatchNorm1d(100)
        self.lstm2, self.bn_l2 = nn.LSTM(100, 50, batch_first=True), nn.BatchNorm1d(50)
        self.att_w, self.att_v = nn.Linear(50, 50), nn.Linear(50, 1, bias=False)
        self.out_dim = EMB_DIM

    def conv_block(self, h, conv, bn, mask):
        h = conv(h.transpose(1, 2)).transpose(1, 2)
        h = F.relu(masked_bn(h, mask, bn))
        return self.drop(h) * mask.unsqueeze(-1)

    def forward(self, x, lengths):
        T = x.shape[1]
        mask = torch.arange(T, device=x.device)[None, :] < lengths.to(x.device)[:, None]
        h = self.conv_block(x, self.conv1, self.bn_c1, mask)
        h = self.conv_block(h, self.conv2, self.bn_c2, mask)
        p = pack_padded_sequence(h, lengths.cpu(), batch_first=True, enforce_sorted=False)
        p, _ = self.lstm1(p)
        p = map_packed(map_packed(p, self.bn_l1), self.drop)
        p, _ = self.lstm2(p)
        p = map_packed(p, self.bn_l2)
        H, _ = pad_packed_sequence(p, batch_first=True, total_length=T)
        e = self.att_v(torch.tanh(self.att_w(H))).squeeze(-1)
        a = torch.softmax(e.masked_fill(~mask, float("-inf")), dim=1)
        return (a.unsqueeze(-1) * H).sum(1), a


class FileBranch(nn.Module):
    def __init__(self, n_in, p=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, 128), nn.BatchNorm1d(128), nn.ReLU(),
                                 nn.Dropout(p), nn.Linear(128, EMB_DIM))
        self.out_dim = EMB_DIM

    def forward(self, x):
        return self.net(x)


class ProsodyClassifier(nn.Module):
    def __init__(self, n_in, p=DROPOUT):
        super().__init__()
        self.branch = SeqBranch(n_in, p)
        self.head = nn.Sequential(nn.Linear(EMB_DIM, 50), nn.ReLU(), nn.Dropout(p), nn.Linear(50, 1))

    def forward(self, xp, lp, xm, xe):
        z, _ = self.branch(xp, lp)
        return self.head(z).squeeze(-1)


class DetectorPTM(nn.Module):
    def __init__(self, ramo, p=DROPOUT):
        super().__init__()
        assert ramo in ("E", "Wf")
        self.ramo = ramo
        if ramo == "E":
            self.eb = FileBranch(D_ECAPA, p)
        else:
            self.wfb = FileBranch(D_WHISPER, p)
        self.head = nn.Sequential(nn.Linear(EMB_DIM, 50), nn.ReLU(), nn.Dropout(p), nn.Linear(50, 1))

    def forward(self, xp, lp, xm, xe):
        z = self.eb(xe) if self.ramo == "E" else self.wfb(xm)
        return self.head(z).squeeze(-1)


class DetectorFull(nn.Module):
    def __init__(self, n_in_p, p=DROPOUT):
        super().__init__()
        self.pb = SeqBranch(n_in_p, p)
        self.wfb = FileBranch(D_WHISPER, p)
        self.eb = FileBranch(D_ECAPA, p)
        self.bn_fuse = nn.ModuleList([nn.BatchNorm1d(EMB_DIM) for _ in range(3)])
        self.head = nn.Sequential(nn.Linear(3 * EMB_DIM, 50), nn.ReLU(), nn.Dropout(p), nn.Linear(50, 1))

    def forward(self, xp, lp, xm, xe):
        zp, _ = self.pb(xp, lp)
        parts = [bn(z) for bn, z in zip(self.bn_fuse, [zp, self.wfb(xm), self.eb(xe)])]
        return self.head(torch.cat(parts, dim=1)).squeeze(-1)


def costruisci(nome):
    if nome == "P":    return ProsodyClassifier(N_CH)
    if nome == "E":    return DetectorPTM("E")
    if nome == "Wf":   return DetectorPTM("Wf")
    if nome == "PWfE": return DetectorFull(N_CH)
    raise ValueError(nome)


def carica_checkpoint(nome, path=None):
    m = costruisci(nome).to(DEVICE)
    m.load_state_dict(torch.load(path or CKPT[nome], map_location=DEVICE), strict=True)
    return m


USA_P = {"P": True, "Wf": False, "E": False, "PWfE": True}    # i modelli Wf ed E non leggono la prosodia
for n in NOMI:
    assert sum(p.numel() for p in costruisci(n).parameters()) == N_PARAM_FT[n], n

# --- preprocessing con le statistiche ORIGINALI dell'addestramento (identico al notebook 11) -----------
PP = json.loads(Path(PREPROC_PROSODIA).read_text())
ST = json.loads(Path(STANDARDIZER).read_text())
assert len(PP["clip_lo"]) == N_CH, "preproc_prosody.json non corrisponde alla variante D (16 canali)"
clip_lo, clip_hi = np.array(PP["clip_lo"], np.float32), np.array(PP["clip_hi"], np.float32)
mm_lo, mm_rng = np.array(PP["minmax_lo"], np.float32), np.array(PP["minmax_range"], np.float32)
w_mu, w_sd = np.array(ST["whisper_file_mean"], np.float32), np.array(ST["whisper_file_std"], np.float32)
e_mu, e_sd = np.array(ST["ecapa_mean"], np.float32), np.array(ST["ecapa_std"], np.float32)


def voiced_mask(x):
    return np.abs(x).sum(1) > 0


def dynamic_features(x):
    v = voiced_mask(x); n = len(x)
    d_ok = np.zeros(n, dtype=bool); d_ok[1:] = v[1:] & v[:-1]
    d = np.zeros_like(x); d[d_ok] = x[d_ok] - x[np.where(d_ok)[0] - 1]
    return d, d_ok


_x = np.array([[180.], [195.], [170.], [0.], [175.], [190.], [185.]], dtype=np.float32)
assert dynamic_features(_x)[0][:, 0].tolist() == [0, 15, -25, 0, 0, 15, -5]


def preprocessa(x):
    d, d_ok = dynamic_features(x)
    xd = np.concatenate([x, d], axis=1)
    m = np.concatenate([np.repeat(voiced_mask(x)[:, None], x.shape[1], 1),
                        np.repeat(d_ok[:, None], x.shape[1], 1)], axis=1)
    xc = np.where(m, np.clip(xd, clip_lo, clip_hi), xd)
    return ((xc - mm_lo) / mm_rng).astype(np.float32)


def costruisci_insieme(seqs, wm, ec, y):
    """Caratteristiche grezze -> insieme pronto per il modello (prosodia impacchettata, PTM standardizzate)."""
    assert len(seqs) == len(wm) == len(ec) == len(y)
    proc = [preprocessa(s) for s in seqs]
    lp = np.array([len(s) for s in proc], dtype=np.int64)
    return {"Xp": np.concatenate(proc).astype(np.float32),
            "off": np.concatenate([[0], np.cumsum(lp)]).astype(np.int64), "lp": lp,
            "wm": ((wm - w_mu) / w_sd).astype(np.float32), "ec": ((ec - e_mu) / e_sd).astype(np.float32),
            "y": np.asarray(y, dtype=np.float32)}


def batch_tensori(D, idx, serve_p):
    if serve_p:
        lp = D["lp"][idx]
        Bp = torch.zeros(len(idx), int(lp.max()), N_CH)
        for k, i in enumerate(idx):
            n = int(D["lp"][i])
            Bp[k, :n] = torch.from_numpy(D["Xp"][D["off"][i]:D["off"][i] + n])
    else:                                   # ingresso non letto dal modello: segnaposto
        lp = np.ones(len(idx), dtype=np.int64)
        Bp = torch.zeros(len(idx), 1, N_CH)
    return (Bp.to(DEVICE), torch.from_numpy(lp),
            torch.from_numpy(D["wm"][idx]).to(DEVICE), torch.from_numpy(D["ec"][idx]).to(DEVICE))


@torch.no_grad()
def punteggi(model, D, serve_p):
    model.eval()
    out = []
    for k in range(0, len(D["y"]), EVAL_BATCH):
        idx = np.arange(k, min(k + EVAL_BATCH, len(D["y"])))
        out.append(model(*batch_tensori(D, idx, serve_p)).float().cpu().numpy())
    return np.concatenate(out).astype(np.float64)


def salva_npz_atomico(f_out, **dati):
    f_tmp = f_out.with_name(f_out.stem + ".tmp.npz")
    np.savez(f_tmp, **dati)
    f_tmp.replace(f_out)


def salva_torch_atomico(obj, f_out):
    f_tmp = f_out.with_name(f_out.name + ".tmp")
    torch.save(obj, f_tmp)
    f_tmp.replace(f_out)

print("modelli e preprocessing pronti")

modelli e preprocessing pronti


## Fase `verifica`

1. **Encoder**: `libmp3lame`, `aac` e `libvorbis` devono essere disponibili nell'ffmpeg della sessione.
2. **Effetto dei codec** su 30 file reali del train: bitrate effettivo, ritardo stimato, SNR residuo. Da
   controllare a mano: `lag` piccolo, SNR finito e crescente con il bitrate.
3. **Test di equivalenza** sui primi 2.000 file del train, cioè il primo blocco del notebook 08, così i batch di
   Whisper ed ECAPA contengono gli stessi file: il nuovo percorso (soundfile → Praat da array; Whisper ed ECAPA
   come nel 08) applicato agli **originali** deve riprodurre le caratteristiche salvate. Criterio fissato prima,
   tutte le condizioni insieme:
   - stesso numero di finestre Praat in **tutti** i file e almeno il 99% dei file identici bit a bit;
   - norma relativa mediana della differenza < 1e-3 per `whisper_mean` e per `ecapa`;
   - scarto mediano dei punteggi < 1e-2 per tutti e quattro i modelli (Sez. 6.7: il massimo non è un criterio).

   Il criterio sui soli punteggi non basta: nel collaudo su dati sintetici uno spostamento sistematico del 5%
   dell'intensità media su tutti i file lo superava.

In [7]:
if "verifica" in ATTIVE:
    # --- 1. encoder -------------------------------------------------------------------------------------
    r = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True)
    print(subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0])
    for c, cfg in CODEC_CFG.items():
        assert re.search(rf"\s{re.escape(cfg['enc'])}\s", r.stdout), f"encoder {cfg['enc']} ({c}) non disponibile"
    print("encoder disponibili:", [cfg["enc"] for cfg in CODEC_CFG.values()])

    # --- 2. effetto dei codec su file reali ------------------------------------------------------------
    g = np.random.default_rng(SEED)
    prove = FILES["train"].iloc[np.sort(g.choice(len(FILES["train"]), N_PROVE_CODEC, replace=False))]
    righe = []
    for codec in CODECS:
        for utt, p in zip(prove["utt_id"], prove["path"]):
            x = leggi_audio(p)
            y, m = codifica(x, codec, utt)
            assert len(y) == len(x) and np.isfinite(y).all(), (codec, utt)
            righe.append({"codec": codec, "utt_id": utt, **m})
    tab_codec = pd.DataFrame(righe)
    tab_codec.to_csv(OUT_DIR / "verifica_codec.csv", index=False)
    print("\n=== effetto dei codec su", N_PROVE_CODEC, "file reali del train ===")
    display(tab_codec.groupby(["codec", "livello"]).agg(
        n=("kbps", "size"), kbps_medio=("kbps", "mean"), kbps_min=("kbps", "min"), kbps_max=("kbps", "max"),
        lag_max=("lag", "max"), snr_medio=("snr", "mean"), snr_min=("snr", "min")).round(2))
    print("Intervalli nominali della Tab. IV (kbps): mp3 80-120 / 220-260, m4a 20-32 / 96-112, "
          "ogg 80-96 / 256-320 (ogg high qui = -q:a 10).")

    # --- 3. test di equivalenza sugli originali --------------------------------------------------------
    blk = FILES["train"].iloc[:BLOCCO]
    t0 = time.perf_counter()
    seq_new, waves, _ = elabora_blocco(blk["path"].tolist(), blk["utt_id"].tolist(), None)
    print(f"\nPraat su {len(blk)} originali: {(time.perf_counter() - t0) / 60:.1f} min")
    seq_old = carica_prosodia("train")[:BLOCCO]
    wm_old, ec_old = carica_ptm("train")
    wm_old, ec_old = wm_old[:BLOCCO], ec_old[:BLOCCO]

    n_diversi = sum(len(a) != len(b) for a, b in zip(seq_old, seq_new))
    uguali = [np.array_equal(a, b) for a, b in zip(seq_old, seq_new)]
    diff_p = max((float(np.abs(a - b).max()) for a, b in zip(seq_old, seq_new) if len(a) == len(b)), default=0.0)
    print(f"Praat: numero di finestre diverso in {n_diversi} file | identici bit a bit {sum(uguali)}/{len(uguali)} "
          f"| scarto massimo {diff_p:.3e}")

    t0 = time.perf_counter()
    whisper_mean_batch, ecapa_batch = carica_estrattori()
    wm_new, ec_new = estrai_ptm(waves, whisper_mean_batch, ecapa_batch)
    print(f"Whisper + ECAPA su {len(blk)} originali: {(time.perf_counter() - t0) / 60:.1f} min")
    rel_med = {}
    for nome_f, a, b in [("whisper_mean", wm_old, wm_new), ("ecapa", ec_old, ec_new)]:
        d = np.abs(a - b)
        rel = np.linalg.norm(a - b, axis=1) / np.maximum(np.linalg.norm(a, axis=1), 1e-12)
        rel_med[nome_f] = float(np.median(rel))
        print(f"{nome_f:12s} scarto max {d.max():.3e} | mediano {np.median(d):.3e} | "
              f"norma relativa: mediana {np.median(rel):.3e}, massima {rel.max():.3e}")
    # effetto della composizione del batch su ECAPA (riempimento con zeri fino al file piu' lungo)
    ec_1 = np.concatenate([ecapa_batch([w]) for w in waves[:64]])
    print(f"ECAPA batch da 1 contro batch da 16 (64 file): scarto max {np.abs(ec_1 - ec_new[:64]).max():.3e}")

    y_blk = blk["y_bonafide"].to_numpy()
    D_old = costruisci_insieme(seq_old, wm_old, ec_old, y_blk)
    D_new = costruisci_insieme(seq_new, wm_new, ec_new, y_blk)
    quota_identici = float(np.mean(uguali))
    ok = (n_diversi == 0 and quota_identici >= QUOTA_PRAAT_IDENTICI
          and all(v < TOLL_REL_PTM for v in rel_med.values()))
    righe_eq = []
    for n in NOMI:
        m = carica_checkpoint(n)
        d = np.abs(punteggi(m, D_old, USA_P[n]) - punteggi(m, D_new, USA_P[n]))
        righe_eq.append({"modello": n, "scarto_max": d.max(), "scarto_mediano": np.median(d),
                         "quota_oltre_1e-3": (d > 1e-3).mean()})
        ok &= bool(np.median(d) < TOLL_MEDIANA)
    eq = pd.DataFrame(righe_eq).set_index("modello")
    eq.to_csv(OUT_DIR / "verifica_equivalenza.csv")
    print("\n=== punteggi: caratteristiche salvate contro ricalcolate (stessi 2.000 originali) ===")
    display(eq)
    (OUT_DIR / "verifica_equivalenza.json").write_text(json.dumps(
        {"n_file": len(blk), "finestre_diverse": int(n_diversi), "praat_identici": int(sum(uguali)),
         "praat_scarto_max": diff_p, "ptm_norma_relativa_mediana": rel_med,
         "punteggi": eq.reset_index().to_dict("records"),
         "criterio": {"finestre_diverse": 0, "quota_praat_identici_min": QUOTA_PRAAT_IDENTICI,
                      "ptm_norma_relativa_mediana_max": TOLL_REL_PTM, "punteggi_scarto_mediano_max": TOLL_MEDIANA},
         "esito": "equivalente" if ok else "NON equivalente"}, indent=2, default=float))
    assert ok, ("Il nuovo percorso NON riproduce le caratteristiche salvate secondo il criterio fissato: "
                "non lanciare l'estrazione. Le copie aumentate differirebbero dagli originali anche per il "
                "percorso di estrazione, non solo per il codec.")
    print("\nEQUIVALENZA VERIFICATA: si puo' lanciare FASE = 'estrazione'.")
else:
    print("fase 'verifica' saltata")

fase 'verifica' saltata


## Fase `estrazione`

Per ogni codec, a blocchi di 2.000 file nell'ordine del protocollo (gli stessi blocchi del notebook 08): codec
e Praat in parallelo sui core, poi Whisper ed ECAPA sulla GPU. Ogni blocco è scritto in modo atomico e, alla
ripresa, viene saltato solo se il suo contenuto (gli `utt_id`) coincide con quello atteso (Sez. 6.10 e 6.11).
L'audio aumentato non viene conservato. Tempo stimato: circa 4 ore per i tre codec.

In [8]:
def f_blocco(codec, k):
    return AUG_DIR / f"aug_{codec}_blk{k:03d}.npz"


N_BLOCCHI = math.ceil(EXPECTED_ROWS["train"] / BLOCCO)

if "estrazione" in ATTIVE:
    f_eq = OUT_DIR / "verifica_equivalenza.json"
    assert f_eq.exists(), ("manca l'esito del test di equivalenza: esegui FASE = 'verifica+estrazione', "
                           "oppure imposta RIPRESA sulla cartella 'aug_lpft' che lo contiene")
    esito = json.loads(f_eq.read_text())["esito"]
    assert esito == "equivalente", f"test di equivalenza: {esito}"
    whisper_mean_batch, ecapa_batch = carica_estrattori()
    tr = FILES["train"]
    t_tot = time.perf_counter()
    for codec in CODECS:
        for k in range(N_BLOCCHI):
            sl = tr.iloc[k * BLOCCO:(k + 1) * BLOCCO]
            f_out = f_blocco(codec, k)
            if f_out.exists():
                z = np.load(f_out, allow_pickle=True)
                assert (z["utt_id"] == sl["utt_id"].to_numpy()).all(), f"{f_out.name}: contenuto inatteso"
                print(f"{codec} blocco {k + 1}/{N_BLOCCHI}: gia' presente, saltato"); continue
            t0 = time.perf_counter()
            seqs, waves, metas = elabora_blocco(sl["path"].tolist(), sl["utt_id"].tolist(), codec)
            t_cpu = time.perf_counter() - t0
            wm, ec = estrai_ptm(waves, whisper_mean_batch, ecapa_batch)
            assert np.isfinite(wm).all() and np.isfinite(ec).all()
            lens = np.array([len(s) for s in seqs], dtype=np.int64)
            salva_npz_atomico(
                f_out, utt_id=sl["utt_id"].to_numpy(), X=np.concatenate(seqs).astype(np.float32),
                offsets=np.concatenate([[0], np.cumsum(lens)]).astype(np.int64),
                whisper_mean=wm, ecapa=ec,
                livello=np.array([m["livello"] for m in metas]), param=np.array([m["param"] for m in metas]),
                kbps=np.array([m["kbps"] for m in metas]), lag=np.array([m["lag"] for m in metas]),
                snr=np.array([m["snr"] for m in metas]))
            del waves
            dt = time.perf_counter() - t0
            print(f"{codec} blocco {k + 1}/{N_BLOCCHI}: {len(sl)} file in {dt / 60:.1f} min "
                  f"(codec + Praat {t_cpu / 60:.1f}) | kbps medio {np.mean([m['kbps'] for m in metas]):.0f} "
                  f"| lag max {max(m['lag'] for m in metas)}", flush=True)
    shutil.rmtree(TMP_DIR, ignore_errors=True); TMP_DIR.mkdir(exist_ok=True)
    print(f"\nfase 'estrazione' terminata in {(time.perf_counter() - t_tot) / 60:.1f} min. "
          "Salva l'output come dataset prima di chiudere la sessione.")
else:
    print("fase 'estrazione' saltata")

fase 'estrazione' saltata


## Fase `addestramento`

Prima di tutto un **controllo di riproduzione**: i checkpoint, con le statistiche di preprocessing originali,
devono dare sul dev 2019 gli EER stampati dai notebook di addestramento. Se il controllo passa, il caricamento
di caratteristiche, preprocessing e pesi è quello dei notebook originali.

In [9]:
def carica_aug(codec):
    """Copie di un codec, riunite e verificate contro l'ordine del protocollo del train."""
    blocchi = [f_blocco(codec, k) for k in range(N_BLOCCHI)]
    manca = [b.name for b in blocchi if not b.exists()]
    assert not manca, f"{codec}: blocchi mancanti {manca[:5]}: la fase 'estrazione' non e' completa"
    seqs, wm, ec, info = [], [], [], []
    for b in blocchi:
        z = np.load(b, allow_pickle=True)
        X, off = z["X"], z["offsets"].astype(np.int64)
        seqs += [X[off[i]:off[i + 1]] for i in range(len(z["utt_id"]))]
        wm.append(z["whisper_mean"]); ec.append(z["ecapa"])
        info.append(pd.DataFrame({"utt_id": z["utt_id"], "livello": z["livello"], "kbps": z["kbps"],
                                  "lag": z["lag"], "snr": z["snr"]}))
    info = pd.concat(info, ignore_index=True)
    assert (info["utt_id"].to_numpy() == FILES["train"]["utt_id"].to_numpy()).all(), f"{codec}: ordine errato"
    return seqs, np.concatenate(wm), np.concatenate(ec), info


if "addestramento" in ATTIVE:
    y_tr = FILES["train"]["y_bonafide"].to_numpy()
    y_dv = FILES["dev"]["y_bonafide"].to_numpy()
    seq_tr, seq_dv = carica_prosodia("train"), carica_prosodia("dev")
    (wm_tr, ec_tr), (wm_dv, ec_dv) = carica_ptm("train"), carica_ptm("dev")
    DEV = costruisci_insieme(seq_dv, wm_dv, ec_dv, y_dv)

    TRAIN = {"controllo": costruisci_insieme(seq_tr, wm_tr, ec_tr, y_tr)}
    seq_a, wm_a, ec_a, info_a = list(seq_tr), [wm_tr], [ec_tr], []
    for codec in CODECS:
        s, w, e, info = carica_aug(codec)
        seq_a += s; wm_a.append(w); ec_a.append(e); info_a.append(info.assign(codec=codec))
    TRAIN["aug"] = costruisci_insieme(seq_a, np.concatenate(wm_a), np.concatenate(ec_a),
                                      np.tile(y_tr, 1 + len(CODECS)))
    del seq_a, wm_a, ec_a
    info_a = pd.concat(info_a, ignore_index=True)
    print("=== copie aumentate: bitrate effettivo per codec e livello ===")
    display(info_a.groupby(["codec", "livello"]).agg(n=("kbps", "size"), kbps_medio=("kbps", "mean"),
                                                     snr_medio=("snr", "mean"), lag_max=("lag", "max")).round(2))

    POS_W = float((y_tr == 0).sum() / (y_tr == 1).sum())
    y_aug = TRAIN["aug"]["y"]
    assert abs(float((y_aug == 0).sum() / (y_aug == 1).sum()) - POS_W) < 1e-9
    print(f"pos_weight {POS_W:.4f} | train controllo {len(TRAIN['controllo']['y'])} | "
          f"train aug {len(TRAIN['aug']['y'])} | dev {len(DEV['y'])}")

    # --- controllo di riproduzione dei checkpoint sul dev ---------------------------------------------
    EER_DEV_CKPT = {}
    for n in NOMI:
        EER_DEV_CKPT[n] = 100 * compute_eer(y_dv, punteggi(carica_checkpoint(n), DEV, USA_P[n]))
        rif = EER_DEV_RIF.get(n)
        print(f"{n:5s} EER dev del checkpoint {EER_DEV_CKPT[n]:.2f}%"
              + (f" | atteso {rif:.2f}%" if rif is not None else " | riferimento non disponibile"))
        if rif is not None:
            assert abs(EER_DEV_CKPT[n] - rif) < TOLL_RIF, (
                f"{n}: il checkpoint non riproduce l'EER del dev ({EER_DEV_CKPT[n]:.2f} contro {rif:.2f}). "
                "Caratteristiche, preprocessing o checkpoint non sono quelli dell'addestramento originale.")
else:
    print("fase 'addestramento' saltata")

=== copie aumentate: bitrate effettivo per codec e livello ===


n  kbps_medio  snr_medio  lag_max
codec livello                                       
m4a   high     12670      107.66      33.00     4093
      low      12710       30.48      16.50     4082
mp3   high     12555      244.96      37.49     4093
      low      12825      101.22      25.41     4082
ogg   high     12674      145.71      36.77     4093
      low      12706       63.70      24.13     4091

pos_weight 8.8372 | train controllo 25380 | train aug 101520 | dev 24844
P     EER dev del checkpoint 11.77% | atteso 11.77%
Wf    EER dev del checkpoint 0.71% | riferimento non disponibile
E     EER dev del checkpoint 9.06% | atteso 9.06%
PWfE  EER dev del checkpoint 1.29% | atteso 1.29%


In [10]:
def moduli_testa(model, nome):
    return [model.bn_fuse, model.head] if nome == "PWfE" else [model.head]


def moduli_rami(model, nome):
    if nome == "P":    return [model.branch]
    if nome == "Wf":   return [model.wfb]
    if nome == "E":    return [model.eb]
    if nome == "PWfE": return [model.pb, model.wfb, model.eb]
    raise ValueError(nome)


def allena_fase(model, nome, D_tr, fase, seed, pos_w):
    """Una fase di LP-FT. LP: solo la testa, rami congelati e in eval(). FT: tutti i parametri."""
    solo_testa = fase == "LP"
    for p in model.parameters():
        p.requires_grad_(not solo_testa)
    if solo_testa:
        for m in moduli_testa(model, nome):
            for p in m.parameters():
                p.requires_grad_(True)
    params = [p for p in model.parameters() if p.requires_grad]
    n_par = sum(p.numel() for p in params)
    atteso = (N_PARAM_LP if solo_testa else N_PARAM_FT)[nome]
    assert n_par == atteso, f"{nome} {fase}: {n_par} parametri addestrabili, attesi {atteso}"
    congelati = moduli_rami(model, nome) if solo_testa else []

    # generatori derivati da (seme, fase): una ripresa che salta LP da' a FT gli stessi numeri
    torch.manual_seed(1000 * seed + FASI_LPFT.index(fase))
    g = np.random.default_rng([SEED, seed, FASI_LPFT.index(fase)])
    opt = torch.optim.Adam(params, lr=LR[fase])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=LR_FACTOR, patience=LR_PATIENCE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_w, device=DEVICE))
    serve_p = USA_P[nome]
    y_t = torch.from_numpy(D_tr["y"])
    n = len(D_tr["y"])
    best = {"eer": np.inf, "epoch": 0, "state": None}
    hist, bad = [], 0
    for ep in range(1, MAX_EPOCHS + 1):
        model.train()
        for m in congelati:
            m.eval()                           # niente dropout e statistiche BatchNorm ferme nei rami congelati
        perm = g.permutation(n)
        tot, cnt = 0.0, 0
        for k in range(0, n, BATCH_SIZE):
            b = perm[k:k + BATCH_SIZE]
            if len(b) < 2:
                continue
            out = model(*batch_tensori(D_tr, b, serve_p))
            loss = loss_fn(out, y_t[b].to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(b); cnt += len(b)
        eer = compute_eer(DEV["y"], punteggi(model, DEV, serve_p))
        hist.append({"epoca": ep, "lr": opt.param_groups[0]["lr"], "loss_train": tot / cnt, "eer_dev": eer})
        sched.step(eer)
        if eer < best["eer"]:
            best = {"eer": eer, "epoch": ep, "state": copy.deepcopy(model.state_dict())}; bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best["state"])
    return best, hist, n_par, ep


if "addestramento" in ATTIVE:
    RES = OUT_DIR / "risultati_dev.csv"
    HIST = OUT_DIR / "storia.csv"
    rows = pd.read_csv(RES).to_dict("records") if RES.exists() else []

    def fatto(tag, fase):
        return (MOD_DIR / f"{tag}_{fase}.pt").exists() and any(r["tag"] == tag and r["fase"] == fase for r in rows)

    t_all = time.perf_counter()
    for cond in CONDIZIONI_FT:                 # prima tutti i controlli (rapidi), poi l'augmentation
        for nome in NOMI:
            for seed in SEEDS:
                tag = f"{nome}_{cond}_s{seed}"
                model = carica_checkpoint(nome)
                for fase in FASI_LPFT:
                    f_pt = MOD_DIR / f"{tag}_{fase}.pt"
                    if fatto(tag, fase):
                        model.load_state_dict(torch.load(f_pt, map_location=DEVICE))
                        print(f"{tag} {fase}: gia' presente, caricato"); continue
                    t0 = time.perf_counter()
                    best, hist, n_par, last = allena_fase(model, nome, TRAIN[cond], fase, seed, POS_W)
                    salva_torch_atomico(model.state_dict(), f_pt)
                    np.save(MOD_DIR / f"{tag}_{fase}_dev.npy", punteggi(model, DEV, USA_P[nome]))
                    pd.DataFrame(hist).assign(tag=tag, fase=fase).to_csv(
                        HIST, mode="a", index=False, header=not HIST.exists())
                    rows = [r for r in rows if not (r["tag"] == tag and r["fase"] == fase)]
                    rows.append({"tag": tag, "modello": nome, "condizione": cond, "seme": seed, "fase": fase,
                                 "eer_dev_%": 100 * best["eer"], "eer_dev_checkpoint_%": EER_DEV_CKPT[nome],
                                 "epoca_migliore": best["epoch"], "ultima_epoca": last,
                                 "param_addestrabili": n_par, "minuti": (time.perf_counter() - t0) / 60})
                    pd.DataFrame(rows).to_csv(RES, index=False)
                    r = rows[-1]
                    print(f"{tag} {fase}: EER dev {r['eer_dev_%']:.2f}% (checkpoint {r['eer_dev_checkpoint_%']:.2f}%) "
                          f"| epoca {best['epoch']}/{last} | {n_par:,} parametri | {r['minuti']:.1f} min", flush=True)
                del model
                if DEVICE == "cuda":
                    torch.cuda.empty_cache()
    print(f"\naddestramento terminato in {(time.perf_counter() - t_all) / 60:.1f} min")
    res = pd.DataFrame(rows)
    display(res.groupby(["modello", "condizione", "fase"])["eer_dev_%"].agg(["mean", "std", "min", "max"]).round(2))
    print("Salva l'output come dataset prima di chiudere la sessione.")

P_controllo_s0 LP: EER dev 11.23% (checkpoint 11.77%) | epoca 10/20 | 2,601 parametri | 2.0 min
P_controllo_s0 FT: EER dev 10.48% (checkpoint 11.77%) | epoca 7/17 | 239,645 parametri | 3.4 min
P_controllo_s1 LP: EER dev 11.42% (checkpoint 11.77%) | epoca 2/12 | 2,601 parametri | 1.1 min
P_controllo_s1 FT: EER dev 10.04% (checkpoint 11.77%) | epoca 19/29 | 239,645 parametri | 5.8 min
P_controllo_s2 LP: EER dev 11.35% (checkpoint 11.77%) | epoca 4/14 | 2,601 parametri | 1.4 min
P_controllo_s2 FT: EER dev 10.20% (checkpoint 11.77%) | epoca 23/33 | 239,645 parametri | 6.7 min
Wf_controllo_s0 LP: EER dev 0.78% (checkpoint 0.71%) | epoca 1/11 | 2,601 parametri | 0.3 min
Wf_controllo_s0 FT: EER dev 0.51% (checkpoint 0.71%) | epoca 39/40 | 74,971 parametri | 1.3 min
Wf_controllo_s1 LP: EER dev 0.78% (checkpoint 0.71%) | epoca 1/11 | 2,601 parametri | 0.3 min
Wf_controllo_s1 FT: EER dev 0.52% (checkpoint 0.71%) | epoca 7/17 | 74,971 parametri | 0.5 min
Wf_controllo_s2 LP: EER dev 0.78% (checkpo

mean   std    min    max
modello condizione fase                           
E       aug        FT     7.91  0.05   7.88   7.97
                   LP     8.66  0.16   8.49   8.79
        controllo  FT     8.84  0.13   8.75   8.99
                   LP     9.38  0.04   9.34   9.41
P       aug        FT     9.64  0.31   9.33   9.96
                   LP    11.32  0.05  11.26  11.35
        controllo  FT    10.24  0.22  10.04  10.48
                   LP    11.33  0.10  11.23  11.42
PWfE    aug        FT     1.08  0.20   0.90   1.30
                   LP     1.32  0.20   1.09   1.45
        controllo  FT     1.02  0.03   0.99   1.06
                   LP     1.13  0.05   1.07   1.17
Wf      aug        FT     0.75  0.00   0.75   0.75
                   LP     0.71  0.00   0.71   0.71
        controllo  FT     0.55  0.07   0.51   0.63
                   LP     0.78  0.00   0.78   0.78

Salva l'output come dataset prima di chiudere la sessione.


## Fase `valutazione`

Su ASVspoof 2021 DF `eval` (533.928 prove, l'unico sottoinsieme confrontabile con la classifica ufficiale).
Prima un **controllo di riproduzione**: i quattro checkpoint devono dare gli EER del notebook 11
(26,83 / 12,35 / 22,09 / 16,09). Poi i punteggi dei 48 modelli (4 modelli × 2 condizioni × 3 semi × 2 fasi),
salvati uno per uno e ripresi se già presenti.

In [11]:
if "valutazione" in ATTIVE:
    COLS = ["speaker", "utt_id", "codec", "source", "attack", "label", "trim", "subset", "vocoder",
            "c9", "c10", "c11", "c12"]
    meta = pd.read_csv(DF_KEYS, sep=r"\s+", header=None, names=COLS, dtype=str, engine="python")
    meta = meta[meta["subset"] == "eval"].reset_index(drop=True)
    Y = (meta["label"] == "bonafide").astype(np.int8).to_numpy()
    assert len(meta) == 533928 and int(Y.sum()) == 14869, (len(meta), int(Y.sum()))
    pos = {u: i for i, u in enumerate(meta["utt_id"])}
    N = len(meta)

    WM = np.full((N, D_WHISPER), np.nan, np.float32); EC = np.full((N, D_ECAPA), np.nan, np.float32)
    for p in sorted(DF_PTM_DIR.glob("ptm_*.npz")):
        z = np.load(p, allow_pickle=True)
        idx = np.array([pos[u] for u in z["utt_id"]])
        WM[idx] = z["whisper_mean"]; EC[idx] = z["ecapa"]
    assert np.isfinite(WM).all() and np.isfinite(EC).all(), "caratteristiche PTM del DF incomplete"
    seqs = [None] * N
    for p in sorted(DF_PROS_DIR.glob("pros_*.npz")):
        z = np.load(p, allow_pickle=True)
        X, off = z["X"], z["offsets"].astype(np.int64)
        for j, u in enumerate(z["utt_id"]):
            seqs[pos[u]] = X[off[j]:off[j + 1]]
    assert all(s is not None for s in seqs), "caratteristiche prosodiche del DF incomplete"
    D_DF = costruisci_insieme(seqs, WM, EC, Y)
    del seqs, WM, EC
    print(f"DF eval: {N:,} prove | bona fide {int(Y.sum()):,}")

    # --- controllo di riproduzione dei checkpoint su eval ----------------------------------------------
    S = {}
    for n in NOMI:
        f_sc = SCO_DIR / f"{n}_checkpoint.npy"
        S[(n, "checkpoint", -1, "-")] = np.load(f_sc) if f_sc.exists() else punteggi(carica_checkpoint(n), D_DF, USA_P[n])
        np.save(f_sc, S[(n, "checkpoint", -1, "-")])
        e = 100 * compute_eer(Y, S[(n, "checkpoint", -1, "-")])
        print(f"{n:5s} checkpoint: EER eval {e:.2f}% | notebook 11: {EER_EVAL_RIF[n]:.2f}%")
        assert abs(e - EER_EVAL_RIF[n]) < TOLL_RIF, f"{n}: il checkpoint non riproduce l'EER del notebook 11"

    # --- punteggi dei modelli LP-FT -------------------------------------------------------------------
    mancanti = []
    for cond in CONDIZIONI_FT:
        for n in NOMI:
            for seed in SEEDS:
                for fase in FASI_LPFT:
                    tag = f"{n}_{cond}_s{seed}"
                    f_sc, f_pt = SCO_DIR / f"{tag}_{fase}.npy", MOD_DIR / f"{tag}_{fase}.pt"
                    if f_sc.exists():
                        S[(n, cond, seed, fase)] = np.load(f_sc); continue
                    if not f_pt.exists():
                        mancanti.append(f"{tag}_{fase}"); continue
                    s = punteggi(carica_checkpoint(n, f_pt), D_DF, USA_P[n])
                    np.save(f_sc, s); S[(n, cond, seed, fase)] = s
    print(f"punteggi disponibili: {len(S) - len(NOMI)} modelli LP-FT"
          + (f" | MANCANTI {len(mancanti)}: {mancanti[:6]}" if mancanti else ""))
else:
    print("fase 'valutazione' saltata")

DF eval: 533,928 prove | bona fide 14,869
P     checkpoint: EER eval 26.83% | notebook 11: 26.83%
Wf    checkpoint: EER eval 12.35% | notebook 11: 12.35%
E     checkpoint: EER eval 22.09% | notebook 11: 22.09%
PWfE  checkpoint: EER eval 16.09% | notebook 11: 16.09%
punteggi disponibili: 48 modelli LP-FT


In [12]:
if "valutazione" in ATTIVE:
    # --- EER di ogni modello e riepilogo sui semi --------------------------------------------------------
    righe = [{"modello": n, "condizione": c, "seme": s, "fase": f, "eer_%": 100 * compute_eer(Y, sc)}
             for (n, c, s, f), sc in S.items()]
    tab = pd.DataFrame(righe)
    tab.to_csv(OUT_DIR / "eer_eval_per_modello.csv", index=False)
    rias = (tab[tab["condizione"] != "checkpoint"].groupby(["modello", "fase", "condizione"])["eer_%"]
            .agg(["mean", "std", "min", "max", "count"]).round(2))
    rias.to_csv(OUT_DIR / "eer_eval_riepilogo.csv")
    print("=== EER (%) su DF eval: media e deviazione standard sui semi ===")
    print("    checkpoint:", {n: round(100 * compute_eer(Y, S[(n, 'checkpoint', -1, '-')]), 2) for n in NOMI})
    display(rias)

    # --- confronto primario: aug contro controllo, appaiato, per seme --------------------------------
    F_CONTR = OUT_DIR / "contrasti_aug_controllo.csv"
    contr = pd.read_csv(F_CONTR).to_dict("records") if F_CONTR.exists() else []
    fatti = {(r["modello"], r["fase"], int(r["seme"])) for r in contr}
    t0 = time.perf_counter()
    for n in NOMI:
        for f in FASI_LPFT:
            for s in SEEDS:
                if (n, f, s) in fatti or (n, "aug", s, f) not in S or (n, "controllo", s, f) not in S:
                    continue
                d, (lo, hi) = bootstrap_appaiato(Y, S[(n, "controllo", s, f)], S[(n, "aug", s, f)])
                contr.append({"modello": n, "fase": f, "seme": s,
                              "eer_controllo_%": 100 * compute_eer(Y, S[(n, "controllo", s, f)]),
                              "eer_aug_%": 100 * compute_eer(Y, S[(n, "aug", s, f)]),
                              "delta_boot": d, "ic_lo": lo, "ic_hi": hi})
                pd.DataFrame(contr).to_csv(F_CONTR, index=False)       # si riprende da qui se interrotto
                print(f"  bootstrap {n} {f} seme {s}: delta {d:+.2f} [{lo:+.2f}, {hi:+.2f}] "
                      f"| {(time.perf_counter() - t0) / 60:.1f} min", flush=True)
    contr = pd.DataFrame(contr, columns=["modello", "fase", "seme", "eer_controllo_%", "eer_aug_%",
                                          "delta_boot", "ic_lo", "ic_hi"])
    contr["delta_%"] = contr["eer_aug_%"] - contr["eer_controllo_%"]
    if contr.empty:
        print("\nNessun confronto aug-controllo disponibile: mancano i punteggi dei modelli LP-FT.")

    def esito(g):
        if len(g) < len(SEEDS):
            return "semi incompleti"
        if (g["ic_hi"] < 0).all():
            return "augmentation migliore"
        if (g["ic_lo"] > 0).all():
            return "augmentation peggiore"
        return "non conclusivo"

    decis = pd.DataFrame(columns=["modello", "fase", "delta_medio_%", "delta_min_%", "delta_max_%", "esito"]) \
        if contr.empty else pd.DataFrame([{"modello": n, "fase": f, "delta_medio_%": g["delta_%"].mean(),
                           "delta_min_%": g["delta_%"].min(), "delta_max_%": g["delta_%"].max(), "esito": esito(g)}
                          for (n, f), g in contr.groupby(["modello", "fase"])]).set_index(["modello", "fase"])
    decis.to_csv(OUT_DIR / "decisione_aug_controllo.csv")
    print("\n=== delta = EER(aug) - EER(controllo), per seme; delta < 0 = augmentation migliore ===")
    display(contr.round(2))
    print("Regola: 'migliore' solo se l'IC 95% e' sotto lo zero in TUTTI i semi (fissata prima dell'esecuzione).")
    display(decis.round(2))

    # --- secondario (descrittivo): quanto cambia il solo addestramento aggiuntivo ----------------------
    rif = {n: 100 * compute_eer(Y, S[(n, "checkpoint", -1, "-")]) for n in NOMI}
    extra = (tab[tab["condizione"] == "controllo"].groupby(["modello", "fase"])["eer_%"].mean()
             .rename("eer_controllo_medio_%").to_frame())
    extra["eer_checkpoint_%"] = [rif[m] for m, _ in extra.index]
    extra["differenza_%"] = extra["eer_controllo_medio_%"] - extra["eer_checkpoint_%"]
    print("\n=== effetto del solo addestramento aggiuntivo (controllo contro checkpoint), descrittivo ===")
    print("    Attenzione: P, Wf ed E originali erano scelti sulla loss del dev, qui sull'EER del dev.")
    display(extra.round(2))

=== EER (%) su DF eval: media e deviazione standard sui semi ===
    checkpoint: {'P': 26.83, 'Wf': 12.35, 'E': 22.09, 'PWfE': 16.09}


mean   std    min    max  count
modello fase condizione                                  
E       FT   aug         24.93  0.31  24.66  25.27      3
             controllo   24.14  0.35  23.83  24.53      3
        LP   aug         24.64  0.71  23.88  25.30      3
             controllo   22.68  0.04  22.64  22.72      3
P       FT   aug         26.68  0.77  26.12  27.56      3
             controllo   25.71  0.54  25.15  26.22      3
        LP   aug         25.77  0.84  25.25  26.73      3
             controllo   25.62  0.49  25.09  26.05      3
PWfE    FT   aug         17.66  0.98  16.53  18.26      3
             controllo   15.50  1.89  13.99  17.63      3
        LP   aug         17.22  2.03  15.81  19.54      3
             controllo   15.69  1.19  14.93  17.06      3
Wf      FT   aug         12.45  0.13  12.30  12.55      3
             controllo   12.04  0.22  11.80  12.20      3
        LP   aug         12.77  0.26  12.52  13.05      3
             controllo   12.47  0.05  12.42  12.53      3

  bootstrap P LP seme 0: delta -0.80 [-0.96, -0.64] | 3.2 min
  bootstrap P LP seme 1: delta +0.22 [+0.10, +0.34] | 6.3 min
  bootstrap P LP seme 2: delta +1.00 [+0.86, +1.15] | 9.4 min
  bootstrap P FT seme 0: delta +2.42 [+2.14, +2.71] | 12.5 min
  bootstrap P FT seme 1: delta +0.38 [+0.14, +0.61] | 15.6 min
  bootstrap P FT seme 2: delta +0.13 [-0.10, +0.36] | 18.7 min
  bootstrap Wf LP seme 0: delta +0.62 [+0.43, +0.81] | 21.8 min
  bootstrap Wf LP seme 1: delta -0.00 [-0.15, +0.15] | 25.0 min
  bootstrap Wf LP seme 2: delta +0.27 [+0.11, +0.43] | 28.1 min
  bootstrap Wf FT seme 0: delta +0.35 [+0.22, +0.48] | 31.2 min
  bootstrap Wf FT seme 1: delta +0.50 [+0.34, +0.67] | 34.3 min
  bootstrap Wf FT seme 2: delta +0.36 [+0.21, +0.52] | 37.4 min
  bootstrap E LP seme 0: delta +2.65 [+2.39, +2.95] | 40.6 min
  bootstrap E LP seme 1: delta +2.09 [+1.85, +2.34] | 43.7 min
  bootstrap E LP seme 2: delta +1.17 [+0.98, +1.36] | 46.9 min
  bootstrap E FT seme 0: delta +1.20 [+0.95, +1.43] 

,modello,fase,seme,eer_controllo_%,eer_aug_%,delta_boot,ic_lo,ic_hi,delta_%
0,P,LP,0,26.05,25.25,-0.80,-0.96,-0.64,-0.80
1,P,LP,1,25.09,25.32,0.22,0.10,0.34,0.24
2,P,LP,2,25.72,26.73,1.00,0.86,1.15,1.02
3,P,FT,0,25.15,27.56,2.42,2.14,2.71,2.41
4,P,FT,1,25.77,26.12,0.38,0.14,0.61,0.36
5,P,FT,2,26.22,26.35,0.13,-0.10,0.36,0.13
6,Wf,LP,0,12.42,13.05,0.62,0.43,0.81,0.63
7,Wf,LP,1,12.53,12.52,-0.00,-0.15,0.15,-0.01
8,Wf,LP,2,12.47,12.74,0.27,0.11,0.43,0.28
9,Wf,FT,0,12.20,12.55,0.35,0.22,0.48,0.35


Regola: 'migliore' solo se l'IC 95% e' sotto lo zero in TUTTI i semi (fissata prima dell'esecuzione).


delta_medio_%  delta_min_%  delta_max_%                  esito
modello fase                                                                
E       FT             0.79         0.13         1.21         non conclusivo
        LP             1.97         1.17         2.63  augmentation peggiore
P       FT             0.97         0.13         2.41         non conclusivo
        LP             0.15        -0.80         1.02         non conclusivo
PWfE    FT             2.16         0.56         3.36  augmentation peggiore
        LP             1.53         0.88         2.49  augmentation peggiore
Wf      FT             0.40         0.35         0.50  augmentation peggiore
        LP             0.30        -0.01         0.63         non conclusivo


=== effetto del solo addestramento aggiuntivo (controllo contro checkpoint), descrittivo ===
    Attenzione: P, Wf ed E originali erano scelti sulla loss del dev, qui sull'EER del dev.


eer_controllo_medio_%  eer_checkpoint_%  differenza_%
modello fase                                                       
E       FT                    24.14             22.09          2.05
        LP                    22.68             22.09          0.59
P       FT                    25.71             26.83         -1.12
        LP                    25.62             26.83         -1.21
PWfE    FT                    15.50             16.09         -0.58
        LP                    15.69             16.09         -0.40
Wf      FT                    12.04             12.35         -0.31
        LP                    12.47             12.35          0.12

In [13]:
if "valutazione" in ATTIVE:
    # --- scomposizione per condizione di codec del DF e per sorgente (media sui semi) -----------------
    def eer_gruppi(colonna, sc):
        out = {}
        for gv in sorted(meta[colonna].unique()):
            m = (meta[colonna] == gv).to_numpy()
            if min(int(Y[m].sum()), int((1 - Y[m]).sum())) >= MIN_PER_CLASSE:
                out[gv] = 100 * compute_eer(Y[m], sc[m])
        return out

    for colonna, titolo in [("codec", "condizione di codec del DF"), ("source", "sorgente")]:
        righe = []
        for n in NOMI:
            righe.append({"modello": n, "configurazione": "checkpoint",
                          **eer_gruppi(colonna, S[(n, "checkpoint", -1, "-")])})
            for c in CONDIZIONI_FT:
                for f in FASI_LPFT:
                    per_seme = [eer_gruppi(colonna, S[(n, c, s, f)]) for s in SEEDS if (n, c, s, f) in S]
                    if per_seme:
                        righe.append({"modello": n, "configurazione": f"{c} {f}",
                                      **pd.DataFrame(per_seme).mean().to_dict()})
        t = pd.DataFrame(righe).set_index(["modello", "configurazione"])
        if colonna == "codec":
            t["escursione"] = t.max(axis=1) - t.min(axis=1)
        t.to_csv(OUT_DIR / f"eer_eval_per_{colonna}.csv")
        print(f"\n=== EER (%) per {titolo} (media sui semi) ===")
        display(t.round(2))

    riepilogo = {
        "dataset": "ASVspoof 2021 DF, sottoinsieme eval (533.928 prove)",
        "disegno": {"riferimento": "checkpoint esistenti, 1 seme", "controllo": "LP-FT sui soli originali",
                    "aug": "LP-FT su originali + 1 copia per codec", "semi": SEEDS, "fasi_lpft": LR,
                    "criterio": "EER del dev 2019 pulito; early stopping pazienza 10, massimo 40 epoche; "
                                "almeno un'epoca (il checkpoint non e' candidato)"},
        "augmentation": {"codec": CODEC_CFG, "frequenza_di_codifica": ENC_SR,
                         "fonte_delle_scelte": "Tab. IV di arXiv:2210.02437 (condizioni del DF), non gli EER su eval"},
        "eer_function": {"source": EVAL_METRICS_URL, "sha256": EVAL_METRICS_SHA256},
        "limiti": [
            "baseline = checkpoint esistenti: differiscono dai modelli LP-FT anche per criterio di selezione "
            "(P, Wf ed E scelti sulla loss del dev) e per numero di semi; l'effetto dell'augmentation si legge "
            "solo nel confronto con il controllo",
            "i tre semi partono dallo stesso checkpoint: misurano la variabilita' del fine-tuning (ordine dei "
            "batch, dropout), non quella dell'inizializzazione",
            "codec e intervalli di bitrate presi dalla Tab. IV, cioe' da una descrizione del test set: eval e' "
            "in parte in distribuzione rispetto alla compressione, come per i partecipanti che hanno usato "
            "codec simili",
            "gli organizzatori hanno usato configurazioni VBR con ffmpeg e sox; qui bitrate richiesto (mp3, "
            "m4a, ogg low) o qualita' massima (ogg high, perche' 256-320 kbps non sono ottenibili in mono), "
            "codifica a 44,1 kHz; il bitrate effettivo e' misurato e salvato",
            "selezione sul dev 2019 pulito, in dominio: puo' favorire i modelli che meglio si adattano al "
            "2019 non compresso",
            "a parita' di epoche il modello aumentato compie quattro volte i passi di gradiente",
            "mp3 e' nel train: la condizione mp3_064 del benchmark di robustezza (notebook 13) e' vicina alla "
            "distribuzione di addestramento dei modelli aumentati",
            "il confronto con T23 (15,64%) e' anacronistico: Whisper e' successivo alla challenge",
        ]}
    (OUT_DIR / "riepilogo.json").write_text(json.dumps(riepilogo, indent=2, default=str))
    print("\nOutput:", sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))


=== EER (%) per condizione di codec del DF (media sui semi) ===


high_m4a  high_mp3  high_ogg  low_m4a  low_mp3  \
modello configurazione                                                   
P       checkpoint         34.91     33.92     21.50    34.80    34.47   
        controllo LP       32.42     31.72     20.73    32.46    32.40   
        controllo FT       32.54     32.19     20.88    32.32    32.69   
        aug LP             32.38     31.92     20.69    32.54    32.69   
        aug FT             34.29     34.33     21.43    34.15    34.47   
Wf      checkpoint         14.09     14.27     10.29    14.21    14.74   
        controllo LP       14.31     14.60     10.36    14.45    14.86   
        controllo FT       14.01     14.42      9.98    13.93    14.49   
        aug LP             15.00     15.04     10.86    14.98    15.58   
        aug FT             14.76     14.98     10.33    14.57    15.06   
E       checkpoint         23.93     23.27     19.98    23.55    24.65   
        controllo LP       24.49     23.91     20.66    24.02    25.26   
        controllo FT       26.37     26.05     21.79    26.06    27.02   
        aug LP             27.68     27.92     21.91    27.49    29.24   
        aug FT             27.88     27.87     22.34    27.75    29.33   
PWfE    checkpoint         19.91     20.27     12.39    20.07    21.49   
        controllo LP       19.43     19.47     12.43    19.14    20.18   
        controllo FT       18.96     19.18     12.60    18.64    19.82   
        aug LP             21.08     21.35     13.61    21.03    21.99   
        aug FT             20.95     20.97     14.74    20.65    21.82   

                        low_ogg  mp3m4a  nocodec  oggm4a  escursione  
modello configurazione                                                
P       checkpoint        21.72   21.63    33.91   21.81       13.41  
        controllo LP      21.65   20.95    31.34   21.63       11.73  
        controllo FT      21.82   21.24    31.92   21.80       11.82  
        aug LP            21.38   20.74    31.52   21.51       12.00  
        aug FT            21.53   21.44    33.88   21.54       13.04  
Wf      checkpoint        10.68   11.09    13.84   10.60        4.45  
        controllo LP      10.71   11.20    14.12   10.72        4.50  
        controllo FT      10.32   10.45    13.67   10.28        4.50  
        aug LP            10.93   11.18    14.61   10.93        4.73  
        aug FT            10.60   10.76    14.59   10.60        4.72  
E       checkpoint        20.80   20.63    23.88   20.41        4.67  
        controllo LP      21.42   21.10    24.34   20.80        4.61  
        controllo FT      22.59   22.64    26.14   21.98        5.23  
        aug LP            22.58   22.84    27.40   22.13        7.34  
        aug FT            22.87   23.33    27.62   22.40        6.99  
PWfE    checkpoint        13.21   13.52    19.46   13.13        9.11  
        controllo LP      12.64   13.11    19.33   12.62        7.75  
        controllo FT      12.86   13.09    18.68   12.70        7.22  
        aug LP            14.19   14.58    20.75   14.24        8.38  
        aug FT            15.18   15.38    20.38   15.06        7.08


=== EER (%) per sorgente (media sui semi) ===


asvspoof  vcc2018  vcc2020
modello configurazione                            
P       checkpoint         14.33    42.78    45.30
        controllo LP       14.40    40.17    42.09
        controllo FT       13.65    42.04    42.96
        aug LP             14.33    41.16    42.36
        aug FT             14.51    44.08    43.78
Wf      checkpoint          4.37    14.21    17.46
        controllo LP        4.50    13.53    17.81
        controllo FT        3.95    14.71    17.01
        aug LP              4.69    15.85    17.90
        aug FT              4.25    16.17    17.43
E       checkpoint         19.32    21.19    27.65
        controllo LP       19.65    21.48    28.41
        controllo FT       20.45    24.00    30.32
        aug LP             19.46    24.63    31.02
        aug FT             18.90    25.86    32.54
PWfE    checkpoint          3.82    23.71    28.19
        controllo LP        3.47    20.93    26.24
        controllo FT        3.11    19.98    24.95
        aug LP              3.32    24.61    27.94
        aug FT              3.38    22.19    26.06


Output: ['contrasti_aug_controllo.csv', 'decisione_aug_controllo.csv', 'eer_eval_per_codec.csv', 'eer_eval_per_modello.csv', 'eer_eval_per_source.csv', 'eer_eval_riepilogo.csv', 'eval_metrics.py', 'riepilogo.json', 'risultati_dev.csv', 'storia.csv', 'verifica_codec.csv', 'verifica_equivalenza.csv', 'verifica_equivalenza.json']
